# TradeSense · Component 03 · XGBoost manipulation-pattern classifier → `models_xgboost_v1`
**Role in the system.** The Autoencoder (v2) is an unsupervised *anomaly* signal. XGBoost is the supervised *pattern* model: it learns, from the Layer 3 v2 training split, which causal combinations of the six Layer 1 features look like the six injected scenario mechanisms, and it explains each alarm with exact TreeSHAP values.

**What this notebook does, in order:** freeze a hashed protocol → build causal features (and optionally the frozen Autoencoder scores) → select one configuration on Layer 3 **validation** from a pre-declared set of 8 → train the final 3-seed ensemble → compare at a **matched real-market alarm burden** (alarm episodes/day on genuine Layer 1 validation) against the Autoencoder, a logistic-regression baseline and a robust-z baseline → **leave-one-scenario-family-out** test of generalisation to unseen mechanisms → SHAP → one unlock of the Layer 3 **test** split.

**Rules baked in**
* Fit: Layer 3 v2 **train** rows only. Early stopping and configuration selection: Layer 3 **validation** only. Test: locked until validation results are saved, then evaluated once for a pre-declared set.
* Thresholds: genuine Layer 1 **validation** only. No threshold, feature or hyper-parameter is chosen on test.
* Negatives (for training *and* evaluation) are bars whose whole feature look-back (15 bars) is untouched by any injection, so rolling features cannot leak the injection into "clean" bars.
* Layer 3 labels describe injection mechanisms in synthetic aggregate data. Results are evidence inside the synthetic world, not proof of detecting real manipulation. Spoofing/layering/liquidity withdrawal are not represented (TGNN/order-book track).
* Nothing frozen is modified; output goes to a staging folder and is renamed only if every test passes.

## 0 · Setup and frozen-input pinning

In [1]:
import os, sys, json, time, math, re, hashlib, shutil, uuid, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import stats
from IPython.display import display
try:
    import xgboost as xgb
except ImportError:
    import subprocess; subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost']); import xgboost as xgb
from sklearn.linear_model import LogisticRegression
import torch, torch.nn as nn

warnings.filterwarnings('ignore', category=FutureWarning); warnings.filterwarnings('ignore', category=RuntimeWarning); warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.width', 230); pd.set_option('display.max_columns', 60); pd.set_option('display.max_rows', 120); pd.set_option('display.float_format', lambda v: f'{v:,.4g}')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L1 = BASE / 'processed_layer1_final_v1'; L3 = BASE / 'processed_layer3_final_v2'; AE1 = BASE / 'models_autoencoder_v1'; AE2 = BASE / 'models_autoencoder_v2'
OUT_FINAL = BASE / 'models_xgboost_v1'; STAGE = BASE / ('.xgb_stage_' + uuid.uuid4().hex[:8])
FROZEN_NAMES = ['processed_layer1_final_v1', 'processed_layer2_final_v3', 'processed_layer2_model_ready_v1', 'processed_layer3_final_v1', 'processed_layer3_final_v2', 'models_autoencoder_v1', 'models_autoencoder_v2']
for p in (L1, L3, AE1, AE2): assert p.is_dir(), f'required input missing: {p}'
assert not OUT_FINAL.exists(), f'REFUSING to overwrite {OUT_FINAL}'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; UTC = lambda: datetime.now(timezone.utc).isoformat(); RUN_STARTED = UTC(); TS = {}; NTHREAD = max(1, os.cpu_count() or 1)
print('base:', BASE, '| xgboost', xgb.__version__, '| threads', NTHREAD, '| AE device', DEVICE)

def sha256_file(p, chunk=8 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
def pin_dir(d): return {str(p.relative_to(d)): {'size': p.stat().st_size, 'mtime_ns': p.stat().st_mtime_ns, 'sha256': sha256_file(p)} for p in sorted(d.rglob('*')) if p.is_file()}
def jsonable(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (pd.Timestamp, datetime)): return o.isoformat()
    return str(o)
def dump_json(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True); path.write_text(json.dumps(obj, indent=2, default=jsonable), encoding='utf-8')
def auc_rank(x, y):
    x = np.asarray(x, float); y = np.asarray(y).astype(int); m = np.isfinite(x); x, y = x[m], y[m]; n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0 or np.nanstd(x) == 0: return float('nan')
    r = stats.rankdata(x); return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))
def avg_precision(x, y):
    x = np.asarray(x, float); y = np.asarray(y).astype(int); m = np.isfinite(x); x, y = x[m], y[m]
    if y.sum() == 0: return float('nan')
    o = np.argsort(-x, kind='mergesort'); yy = y[o]; tp = np.cumsum(yy); return float(((tp / np.arange(1, len(yy) + 1)) * yy).sum() / yy.sum())
def wilson(k, n, z=1.96):
    if n == 0: return [float('nan'), float('nan')]
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d; return [max(0.0, c - h), min(1.0, c + h)]

t0 = time.time(); PIN = {n: pin_dir(BASE / n) for n in FROZEN_NAMES if (BASE / n).is_dir()}
print('pinned:', {n: len(v) for n, v in PIN.items()}, f'({time.time()-t0:.0f}s)')

Mounted at /content/drive
base: /content/drive/MyDrive/research datasets | xgboost 3.4.1 | threads 2 | AE device cuda
pinned: {'processed_layer1_final_v1': 400, 'processed_layer2_final_v3': 8, 'processed_layer2_model_ready_v1': 27, 'processed_layer3_final_v1': 5, 'processed_layer3_final_v2': 32, 'models_autoencoder_v1': 39, 'models_autoencoder_v2': 47} (134s)


## 1 · FREEZE THE PROTOCOL (before any feature, model or score exists)

In [2]:
FEATURES = ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'rush_order_density', 'volume_acceleration', 'large_trade_count']
SYMS = {'BTCUSDT': 'btc', 'ETHUSDT': 'eth'}; SPL = ('train', 'val', 'test'); WARM = 15; N_BARS = 129600; BOUNDS = {'train': (0, 90720), 'val': (90720, 109440), 'test': (109440, 129600)}; SEEDS = [11, 22, 33]
BASE_FAMS = ['pump_and_dump_like', 'fake_breakout_like', 'abnormal_volume_burst', 'trade_flow_imbalance', 'wash_like_aggregate_proxy', 'liquidity_stress_proxy']
PROTOCOL = {
 'protocol_id': 'xgboost_protocol_v1', 'statement': 'Written and hashed before any feature, model or score of this notebook exists.',
 'labels': {'positive': 'Layer 3 v2 synthetic_label == 1 (bars inside an injected event)', 'negative': 'clean: normal_background == 1 AND no non-clean bar (event, dependency tail, warm-up) in [t-15, t] — so every look-back feature and the Autoencoder context of a negative bar is untouched by any injection',
            'excluded': 'dependency-tail bars and the 15 bars after them; bars < 30', 'note': 'labels describe injection mechanisms, not verified manipulation'},
 'features': {'set_A': 'causal rolling features of the six scaled Layer 1 features (look-back <= 15 bars): z, 1-bar diff, mean3, mean15, absmax3, absmax15, std15, deviation from previous-15 mean (8 x 6 = 48) + 5 cross-feature (number of |z|>3, sum |z|, their short means/max) = 53',
              'set_B': 'set_A + 8 current-bar Autoencoder columns from the frozen Autoencoder v2 recommendation (score fmax, score fmean, 6 per-feature errors / TRAIN p99 scale); AE trained on Layer 1 TRAIN only',
              'scaling': 'robust scaling fitted on Layer 1 TRAIN only (identical to the Autoencoder preprocessing)'},
 'candidates': {'feature_set': ['A', 'B'], 'scope': ['pooled', 'per_symbol'], 'max_depth': [3, 5], 'n_configs': 8},
 'fixed_hyperparameters': {'objective': 'binary:logistic', 'eval_metric': 'aucpr', 'tree_method': 'hist', 'eta': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 5, 'reg_lambda': 1.0, 'scale_pos_weight': 5.0, 'max_rounds': 800, 'early_stopping_rounds': 40},
 'data_use': {'fit': 'Layer 3 v2 train rows (positive or clean)', 'early_stopping_and_selection': 'Layer 3 v2 validation rows / events', 'test': 'locked; unlocked once after validation results are saved; declared set only', 'selection_seed': 11, 'final_seeds': SEEDS, 'score': 'mean of the seeds\' raw margins (log-odds); probability = sigmoid(score), uncalibrated'},
 'burden': {'definition': 'alarm episodes per day on genuine Layer 1 VALIDATION bars (alarm bars merged when gap <= 5 bars); threshold = lowest on a quantile grid whose episodes/day <= budget', 'budgets': [1, 2, 4], 'primary': 2},
 'selection_rule': {'metric': 'mean over BTC/ETH of Layer 3 validation event recall at the primary budget', 'tolerance': 0.05, 'tie_break': 'among configs within tolerance of the best: simplest = (feature_set A before B, pooled before per_symbol, smaller depth), then higher validation PR-AUC'},
 'comparators_declared': ['xgb_final', 'ae_recommended (Autoencoder v2 recommendation, scores recomputed from its models)', 'logreg_A (logistic regression on set A, class weight 5)', 'rz (robust-z baseline)'],
 'lofo': {'definition': 'for each of the 6 base families: retrain the selected configuration without any event containing that family (composites containing it removed too; early-stopping set also without it), threshold at the primary budget on genuine Layer 1 validation, and measure recall on the held-out family\'s non-composite events from the train and validation splits only (test stays locked)', 'seed': 11, 'comparison': 'Autoencoder recommendation on the same events'},
 'shap': {'method': 'exact TreeSHAP (xgboost pred_contribs) averaged over the 3 seeds, on validation positive bars + 3000 random clean validation bars per symbol; reported by feature group and by scenario family; descriptive'},
 'bootstrap': {'B': 200, 'seed': 7}, 'verdict_rule': {'outperforms': 'mean recall at the primary budget exceeds the Autoencoder by >= 0.05 on BOTH validation and test', 'comparable': 'both differences within +/-0.05', 'lofo_strength': 'mean held-out-family recall >= 0.6 strong, >= 0.3 moderate, else weak'},
 'claims_policy': 'Scores are pattern signals on synthetic mechanisms; no claim of detecting real manipulation or intent.'}
PJ = json.dumps(PROTOCOL, sort_keys=True, separators=(',', ':'), ensure_ascii=False); PROTOCOL_HASH = hashlib.sha256(PJ.encode()).hexdigest()
STAGE.mkdir(parents=True, exist_ok=False)
for sub in ('protocol', 'models', 'scores', 'reports', 'validation', 'preprocessing'): (STAGE / sub).mkdir()
dump_json({**PROTOCOL, 'integrity': {'sha256_of_canonical_json_excluding_this_field': PROTOCOL_HASH, 'frozen_utc': UTC()}}, STAGE / 'protocol' / 'xgboost_protocol.json')
TS['protocol_frozen'] = UTC(); print('PROTOCOL FROZEN  sha256 =', PROTOCOL_HASH)

PROTOCOL FROZEN  sha256 = 82e61ccb50c3cc0da6f4373b7260df9dd24c95ea475fe58a00331eb7d2ce9509


## 2 · Load Layer 1 (genuine) and Layer 3 v2; preprocessing identical to the Autoencoder; clean/positive masks

In [3]:
F1, DT, F3, LB3 = {}, {}, {}, {}
for sym, tag in SYMS.items():
    d = pd.read_parquet(L1 / 'processed_matrices' / f'{sym.lower()}_l1_feature_matrix_90d.parquet'); d['datetime'] = pd.to_datetime(d['datetime'], utc=True); d = d.sort_values('datetime').reset_index(drop=True)
    assert len(d) == N_BARS and all(f in d.columns for f in FEATURES); F1[sym] = d[FEATURES].to_numpy(np.float64); DT[sym] = d['datetime']
    fe = pd.concat([pd.read_parquet(L3 / tag / f'{sp}_features.parquet') for sp in SPL], ignore_index=True); lb = pd.concat([pd.read_parquet(L3 / tag / f'{sp}_labels.parquet') for sp in SPL], ignore_index=True)
    assert list(fe.columns) == ['datetime'] + FEATURES and len(fe) == N_BARS and (pd.to_datetime(fe.datetime, utc=True).reset_index(drop=True) == DT[sym]).all() and (pd.to_datetime(lb.datetime, utc=True).reset_index(drop=True) == DT[sym]).all()
    F3[sym] = fe[FEATURES].to_numpy(np.float64); LB3[sym] = lb
EV3 = pd.concat([pd.read_parquet(L3 / 'events' / f'{t}_events.parquet') for t in SYMS.values()], ignore_index=True); EV3['fams'] = EV3.families_json.apply(json.loads)
def transform(F):
    X = np.asarray(F, np.float64).copy(); X[:, 3] = np.log1p(np.maximum(X[:, 3], 0)); X[:, 5] = np.log1p(np.maximum(X[:, 5], 0)); return X
PREP, Z1, Z3 = {}, {}, {}
for sym, tag in SYMS.items():
    X = transform(F1[sym]); tr = X[WARM:BOUNDS['train'][1]]; med = np.median(tr, axis=0); q75, q25 = np.percentile(tr, [75, 25], axis=0); sc = (q75 - q25) / 1.349
    sc = np.maximum(np.where(sc > 1e-12, sc, np.maximum(tr.std(axis=0), 1e-8)), 1e-8); PREP[sym] = {'median': med, 'scale': sc, 'fit_rows': [WARM, BOUNDS['train'][1]]}
    for d_ in (AE1, AE2):
        pp = json.loads((d_ / 'preprocessing' / f'{tag}_preprocessing.json').read_text()); assert np.allclose(pp['median'], med, rtol=1e-9, atol=1e-12) and np.allclose(pp['scale'], sc, rtol=1e-9, atol=1e-12), 'preprocessing differs from the Autoencoder'
    Z1[sym] = np.clip((X - med) / sc, -20, 20).astype(np.float32); Z3[sym] = np.clip((transform(F3[sym]) - med) / sc, -20, 20).astype(np.float32)
LOOKBACK = 15; FEAT_START = 30
POS, CLEAN = {}, {}
for sym in SYMS:
    lb = LB3[sym]; POS[sym] = lb.synthetic_label.to_numpy() == 1; nonclean = (lb.normal_background.to_numpy() == 0).astype(np.int64); c = np.concatenate([[0], np.cumsum(nonclean)]); idx = np.arange(N_BARS)
    CLEAN[sym] = ((c[idx + 1] - c[np.maximum(idx - LOOKBACK, 0)]) == 0) & (idx >= FEAT_START); assert not (CLEAN[sym] & POS[sym]).any()
    print(sym, 'positive bars', {sp: int(POS[sym][a:b].sum()) for sp, (a, b) in BOUNDS.items()}, '| clean bars', {sp: int(CLEAN[sym][a:b].sum()) for sp, (a, b) in BOUNDS.items()})
print('events per split:', EV3.groupby(['symbol', 'split']).size().unstack().to_dict('index')); TS['data_loaded'] = UTC()

BTCUSDT positive bars {'train': 2682, 'val': 891, 'test': 1672} | clean bars {'train': 85608, 'val': 16929, 'test': 17408}
ETHUSDT positive bars {'train': 2211, 'val': 845, 'test': 1259} | clean bars {'train': 86079, 'val': 16975, 'test': 17821}
events per split: {'BTCUSDT': {'test': 36, 'train': 80, 'val': 30}, 'ETHUSDT': {'test': 36, 'train': 80, 'val': 30}}


## 3 · Causal features (set A) and the frozen Autoencoder columns (set B)
Every feature at bar *t* uses bars `t-15..t` only (rolling windows end at *t*), so a negative bar whose `[t-15, t]` is untouched has features identical to the genuine Layer 1 bar (verified in the tests).

In [4]:
def build_feats(Z):
    df = pd.DataFrame(np.asarray(Z, np.float64), columns=FEATURES); cols = {}; prev15 = df.shift(1).rolling(15).mean()
    for f in FEATURES:
        s = df[f]; a = s.abs()
        cols[f + '__z'] = s; cols[f + '__d1'] = s - s.shift(1); cols[f + '__mean3'] = s.rolling(3).mean(); cols[f + '__mean15'] = s.rolling(15).mean(); cols[f + '__amax3'] = a.rolling(3).max()
        cols[f + '__amax15'] = a.rolling(15).max(); cols[f + '__std15'] = s.rolling(15).std(); cols[f + '__dev'] = s - prev15[f]
    A = df.abs(); ne = (A > 3).sum(axis=1).astype(float); sa = A.sum(axis=1)
    cols['x__n_extreme'] = ne; cols['x__sumabs'] = sa; cols['x__n_extreme_mean5'] = ne.rolling(5).mean(); cols['x__sumabs_mean5'] = sa.rolling(5).mean(); cols['x__sumabs_max15'] = sa.rolling(15).max()
    out = pd.DataFrame(cols).to_numpy(np.float32, copy=True); out[:FEAT_START] = np.nan; return out, list(cols)
FA_NAMES = build_feats(Z1['BTCUSDT'][:200])[1]; print(len(FA_NAMES), 'set-A features')

# ---- frozen Autoencoder v2 recommendation (scores recomputed from its models; nothing is retrained) ----
SEL_AE = json.loads((AE2 / 'models' / 'selected_model.json').read_text()); AE_SCORER = SEL_AE['recommended']['scorer']; AE_FAM = AE_SCORER.split('|')[0]; AE_FEAT = AE_SCORER.split('|')[2]
assert AE_FAM in ('ctx', 'dn', 'sb'), f'unsupported Autoencoder family {AE_FAM}'
from numpy.lib.stride_tricks import sliding_window_view
class AE(nn.Module):
    def __init__(s, d, b):
        super().__init__()
        s.enc = nn.Sequential(nn.Linear(d, 64), nn.LeakyReLU(0.1), nn.Linear(64, 32), nn.LeakyReLU(0.1), nn.Linear(32, b)); s.dec = nn.Sequential(nn.Linear(b, 32), nn.LeakyReLU(0.1), nn.Linear(32, 64), nn.LeakyReLU(0.1), nn.Linear(64, d))
    def forward(s, x): return s.dec(s.enc(x))
class CAE(nn.Module):
    def __init__(s, d_ctx, b=3):
        super().__init__(); s.d_ctx = d_ctx
        s.enc = nn.Sequential(nn.Linear(6 + d_ctx, 64), nn.LeakyReLU(0.1), nn.Linear(64, 32), nn.LeakyReLU(0.1), nn.Linear(32, b)); s.dec = nn.Sequential(nn.Linear(b + d_ctx, 32), nn.LeakyReLU(0.1), nn.Linear(32, 64), nn.LeakyReLU(0.1), nn.Linear(64, 6))
    def forward(s, x, c=None):
        if s.d_ctx == 0: return s.dec(s.enc(x))
        return s.dec(torch.cat([s.enc(torch.cat([x, c], 1)), c], 1))
def ctx_feats(Z):
    n = len(Z); C = np.full((n, 12), np.nan, np.float32); win = sliding_window_view(Z.astype(np.float64), (15, 6))[:, 0]
    C[15:] = np.concatenate([win[:n - 15].mean(axis=1), win[:n - 15].std(axis=1)], axis=1).astype(np.float32); return C
AE_FIRST = {'sb': WARM, 'dn': WARM, 'ctx': WARM + 15}[AE_FAM]; AEM = {}
for sym, tag in SYMS.items():
    ms = []
    for sd in SEEDS:
        if AE_FAM == 'sb':
            a = json.loads((AE1 / 'models' / tag / 'single_bar' / 'arch.json').read_text()); m = AE(a['d'], a['bottleneck']); m.load_state_dict(torch.load(AE1 / 'models' / tag / 'single_bar' / f'seed{sd}.pt', map_location='cpu'))
        else:
            dn = {'ctx': 'ctx_bar', 'dn': 'dn_bar'}[AE_FAM]; a = json.loads((AE2 / 'models' / tag / dn / 'arch.json').read_text()); m = CAE(a['d_ctx'], a['bottleneck']); m.load_state_dict(torch.load(AE2 / 'models' / tag / dn / f'seed{sd}.pt', map_location='cpu'))
        ms.append(m.to(DEVICE).eval())
    AEM[sym] = ms
@torch.no_grad()
def ae_err(sym, Z):
    n = len(Z); C = ctx_feats(Z) if AE_FAM == 'ctx' else None; out = np.zeros((n, 6), np.float32)
    for i in range(0, n, 8192):
        x = torch.tensor(Z[i:i + 8192], device=DEVICE); c = None if C is None else torch.tensor(np.nan_to_num(C[i:i + 8192]), device=DEVICE); acc = 0
        for m in AEM[sym]: acc = acc + ((m(x) - x) ** 2 if AE_FAM == 'sb' else (m(x, c) - x) ** 2)
        out[i:i + 8192] = (acc / len(AEM[sym])).cpu().numpy()
    out[:AE_FIRST] = np.nan; return out
AE_NORM = {}
for sym in SYMS:
    E = ae_err(sym, Z1[sym]); AE_NORM[sym] = np.maximum(np.percentile(E[AE_FIRST:BOUNDS['train'][1]].reshape(-1, 6), 99, axis=0), 1e-8)
    saved = SEL_AE['recommended'].get('feature_error_scale_train_p99')
    if saved: assert np.allclose(saved[SYMS[sym]], AE_NORM[sym], rtol=1e-4), 'AE feature-error scale differs from the saved one'
def ae_cols(sym, Z):
    E = ae_err(sym, Z); n_ = E / AE_NORM[sym]; cols = np.column_stack([n_.max(axis=1), E.mean(axis=1), n_]).astype(np.float32); cols[:FEAT_START] = np.nan; return cols
FB_NAMES = FA_NAMES + ['ae__fmax', 'ae__fmean'] + ['ae__e_' + f for f in FEATURES]
XF = {}                                         # (src, sym) -> (n, 53 + 8) float32 ; A = first 53 columns
for src, Zd in (('l1', Z1), ('l3', Z3)):
    for sym in SYMS: XF[(src, sym)] = np.concatenate([build_feats(Zd[sym])[0], ae_cols(sym, Zd[sym])], axis=1)
NA = len(FA_NAMES); AE_SCORE_COL = NA + (0 if AE_FEAT == 'fmax' else 1)
print('features built: set A', NA, '| set B', len(FB_NAMES), '| AE comparator:', AE_SCORER); TS['features_built'] = UTC()
# AE comparator scores are the AE columns themselves: ae__fmax or ae__fmean
SC1, SC3 = {}, {}
for sym in SYMS:
    for src, store in (('l1', SC1), ('l3', SC3)):
        store[(sym, 'ae_recommended')] = XF[(src, sym)][:, AE_SCORE_COL].copy()
        Zd = Z1 if src == 'l1' else Z3; z = np.abs(Zd[sym]).max(axis=1).astype(np.float32); z[:WARM] = np.nan; store[(sym, 'rz')] = z

53 set-A features
features built: set A 53 | set B 61 | AE comparator: ctx|cur|fmax


## 4 · Matched alarm burden, evaluation functions (Layer 3 test stays locked)

In [5]:
EP_GAP = 5; VAL = slice(*BOUNDS['val']); BUDGETS = PROTOCOL['burden']['budgets']; PB = PROTOCOL['burden']['primary']
def episodes(alarm, gap=EP_GAP):
    idx = np.flatnonzero(alarm); return 0 if len(idx) == 0 else int(1 + (np.diff(idx) > gap).sum())
PGRID = 1 - np.logspace(-4.6, -0.3, 260)
def thr_for_budget(v, budget):
    v = v[np.isfinite(v)]; days = len(v) / 1440; cand = np.unique(np.quantile(v, PGRID)); ok = [t for t in cand if episodes(v > t) / days <= budget]
    return float(min(ok)) if ok else float(v.max())
THRB = {}
def register(scorers):
    for sym in SYMS:
        for sc in scorers: THRB[(sym, sc)] = {b: thr_for_budget(SC1[(sym, sc)][VAL], b) for b in BUDGETS}
register(['ae_recommended', 'rz'])
TEST_STATE = {'unlocked': False, 'unlock_utc': None, 'calls': 0}
def l3_eval(sym, split, scorer, ci=False, events=None):
    if split == 'test': assert TEST_STATE['unlocked'], 'test split is locked'; TEST_STATE['calls'] += 1
    scores = SC3[(sym, scorer)]; ops = {f'b{b}': t for b, t in THRB[(sym, scorer)].items()}; a, b = BOUNDS[split]; lb = LB3[sym]; idx = np.arange(N_BARS); ev = EV3[(EV3.symbol == sym) & (EV3.split == split)].reset_index(drop=True)
    ok = (idx >= a) & (idx < b) & np.isfinite(scores); pos = ok & POS[sym]; neg = ok & CLEAN[sym]; m = pos | neg; y = pos[m].astype(int); s = scores[m]
    res = {'n_events': int(len(ev)), 'n_pos_bars': int(pos.sum()), 'n_neg_bars': int(neg.sum()), 'roc_auc': auc_rank(s, y), 'pr_auc': avg_precision(s, y), 'pr_auc_chance': float(y.mean())}
    if ci:
        rng = np.random.default_rng(PROTOCOL['bootstrap']['seed']); evb = [np.arange(r.start_idx, r.end_idx + 1) for r in ev.itertuples()]; negs = scores[neg]; ba, bp = [], []
        for _ in range(PROTOCOL['bootstrap']['B']):
            pick = rng.integers(0, len(ev), len(ev)); ps = np.concatenate([scores[evb[i]] for i in pick]); ps = ps[np.isfinite(ps)]; yy = np.r_[np.ones(len(ps), int), np.zeros(len(negs), int)]; xx = np.r_[ps, negs]; ba.append(auc_rank(xx, yy)); bp.append(avg_precision(xx, yy))
        res['roc_auc_ci95'] = [float(np.nanpercentile(ba, 2.5)), float(np.nanpercentile(ba, 97.5))]; res['pr_auc_ci95'] = [float(np.nanpercentile(bp, 2.5)), float(np.nanpercentile(bp, 97.5))]
    res['ops'] = {}
    for name, t in ops.items():
        al = scores > t; det = np.array([bool(al[r.start_idx:r.end_idx + 1].any()) for r in ev.itertuples()]); delay = np.array([int(np.argmax(al[r.start_idx:r.end_idx + 1])) if d_ else -1 for r, d_ in zip(ev.itertuples(), det)])
        negal = np.where(neg, al, False); days = max(neg.sum(), 1) / 1440; e2 = ev.assign(detected=det); fam = {k: [int(g.detected.sum()), len(g)] for k, g in e2.groupby('scenario_family')}
        res['ops'][name] = {'threshold': float(t), 'event_recall': float(det.mean()), 'event_recall_ci95': wilson(int(det.sum()), len(det)), 'events_detected': int(det.sum()), 'median_detection_delay_bars': float(np.median(delay[det])) if det.any() else None,
            'clean_negative_episodes_per_day': float(episodes(negal) / days), 'bar_recall': float(al[pos].mean()) if pos.any() else None, 'by_family': fam, 'families_covered': int(sum(1 for v in fam.values() if v[0] > 0)), 'families_total': len(fam),
            'by_severity': {k: [int(g.detected.sum()), len(g)] for k, g in e2.groupby('severity')}, 'by_ood': {str(k): [int(g.detected.sum()), len(g)] for k, g in e2.groupby('is_ood')}}
    res['layer1'] = {}
    for name, t in ops.items():
        v = SC1[(sym, scorer)][a:b]; al = v > t; res['layer1'][name] = {'alarm_bars_per_day': float(al.sum() / ((b - a) / 1440)), 'episodes_per_day': float(episodes(al) / ((b - a) / 1440))}
    return res

## 5 · Training function and the 8-configuration selection on Layer 3 VALIDATION

In [6]:
PARAMS = {'objective': 'binary:logistic', 'eval_metric': 'aucpr', 'tree_method': 'hist', 'eta': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 5, 'reg_lambda': 1.0, 'scale_pos_weight': 5.0, 'nthread': NTHREAD}
FP = PROTOCOL['fixed_hyperparameters']; NCOL = {'A': NA, 'B': len(FB_NAMES)}; NAMES = {'A': FA_NAMES, 'B': FB_NAMES}; ROWLOG = []
def rows(sym, split, fs, excl=None, tag=''):
    a, b = BOUNDS[split]; idx = np.arange(N_BARS); m = (idx >= a) & (idx < b) & (POS[sym] | CLEAN[sym]) & (idx >= FEAT_START)
    if excl is not None: m &= ~excl[sym]
    ii = np.flatnonzero(m); ROWLOG.append({'tag': tag, 'sym': sym, 'split': split, 'min_idx': int(ii.min()), 'max_idx': int(ii.max()), 'n': int(len(ii)), 'n_pos': int(POS[sym][ii].sum())})
    return XF[('l3', sym)][ii][:, :NCOL[fs]], POS[sym][ii].astype(np.float32)
BEST_IT = {}
def train_booster(fs, syms, depth, seed, excl=None, tag=''):
    parts = [rows(s, 'train', fs, excl, tag + '|fit') for s in syms]; ves = [rows(s, 'val', fs, excl, tag + '|es') for s in syms]
    Xt, yt = np.concatenate([p[0] for p in parts]), np.concatenate([p[1] for p in parts]); Xv, yv = np.concatenate([p[0] for p in ves]), np.concatenate([p[1] for p in ves])
    dtr = xgb.DMatrix(Xt, label=yt, feature_names=NAMES[fs]); dva = xgb.DMatrix(Xv, label=yv, feature_names=NAMES[fs])
    bst = xgb.train({**PARAMS, 'max_depth': depth, 'seed': seed}, dtr, num_boost_round=FP['max_rounds'], evals=[(dva, 'val')], early_stopping_rounds=FP['early_stopping_rounds'], verbose_eval=False)
    return bst, int(bst.best_iteration), {'n_train': int(len(yt)), 'n_train_pos': int(yt.sum()), 'n_es': int(len(yv)), 'n_es_pos': int(yv.sum()), 'best_iteration': int(bst.best_iteration), 'best_aucpr_val': float(bst.best_score)}
def margin(bsts, fs, X):
    dm = xgb.DMatrix(X[:, :NCOL[fs]], feature_names=NAMES[fs]); return np.mean([b.predict(dm, output_margin=True, iteration_range=(0, bi + 1)) for b, bi in bsts], axis=0).astype(np.float32)
def score_into(name, fs, models):
    for sym in SYMS:
        for src, store in (('l1', SC1), ('l3', SC3)):
            s = margin(models[sym], fs, XF[(src, sym)]); s[:FEAT_START] = np.nan; store[(sym, name)] = s
CONFIGS = [(fs, sc, d) for fs in ('A', 'B') for sc in ('pooled', 'per_symbol') for d in (3, 5)]; cname = lambda c: f'xgb|{c[0]}|{c[1]}|d{c[2]}'
SELROWS, TRAIN_LOG, CFGMODELS = [], [], {}
TS['selection_start'] = UTC(); t0 = time.time()
for c in CONFIGS:
    fs, scope, depth = c; models = {}
    if scope == 'pooled':
        b, bi, info = train_booster(fs, list(SYMS), depth, 11, tag=cname(c)); models = {s: [(b, bi)] for s in SYMS}; TRAIN_LOG.append({'config': cname(c), 'scope': 'pooled', 'seed': 11, **info})
    else:
        for s in SYMS:
            b, bi, info = train_booster(fs, [s], depth, 11, tag=cname(c) + '|' + s); models[s] = [(b, bi)]; TRAIN_LOG.append({'config': cname(c), 'scope': s, 'seed': 11, **info})
    CFGMODELS[c] = models; score_into(cname(c), fs, models); register([cname(c)])
    ev = {s: l3_eval(s, 'val', cname(c)) for s in SYMS}
    SELROWS.append({'config': cname(c), 'feature_set': fs, 'scope': scope, 'max_depth': depth, 'simp': (0 if fs == 'A' else 1, 0 if scope == 'pooled' else 1, depth), 'val_recall_btc': ev['BTCUSDT']['ops'][f'b{PB}']['event_recall'], 'val_recall_eth': ev['ETHUSDT']['ops'][f'b{PB}']['event_recall'],
                    'val_pr_auc': float(np.mean([ev[s]['pr_auc'] for s in SYMS])), 'rounds': [m[0][1] + 1 for m in models.values()]})
    print(f"{cname(c):22s} val recall@b{PB} BTC {SELROWS[-1]['val_recall_btc']:.2f} ETH {SELROWS[-1]['val_recall_eth']:.2f} PR {SELROWS[-1]['val_pr_auc']:.3f} rounds {SELROWS[-1]['rounds']} ({time.time()-t0:.0f}s)")
SELTAB = pd.DataFrame(SELROWS); SELTAB['primary'] = (SELTAB.val_recall_btc + SELTAB.val_recall_eth) / 2
def select_rule(tab):
    best = tab.primary.max(); tied = tab[tab.primary >= best - PROTOCOL['selection_rule']['tolerance']].copy(); tied['k'] = tied.simp.apply(tuple); tied['npr'] = -tied.val_pr_auc
    return tied.sort_values(['k', 'npr']).iloc[0].config
SELECTED = select_rule(SELTAB); SEL_CFG = [c for c in CONFIGS if cname(c) == SELECTED][0]; TS['selection_done'] = UTC()
display(SELTAB.drop(columns='simp').set_index('config')); print('SELECTED (validation only):', SELECTED)

xgb|A|pooled|d3        val recall@b2 BTC 0.50 ETH 0.43 PR 0.355 rounds [28, 28] (6s)
xgb|A|pooled|d5        val recall@b2 BTC 0.57 ETH 0.53 PR 0.404 rounds [15, 15] (9s)
xgb|A|per_symbol|d3    val recall@b2 BTC 0.53 ETH 0.47 PR 0.422 rounds [439, 240] (26s)
xgb|A|per_symbol|d5    val recall@b2 BTC 0.53 ETH 0.43 PR 0.394 rounds [19, 251] (38s)
xgb|B|pooled|d3        val recall@b2 BTC 0.70 ETH 0.63 PR 0.499 rounds [698, 698] (74s)
xgb|B|pooled|d5        val recall@b2 BTC 0.70 ETH 0.60 PR 0.508 rounds [587, 587] (111s)
xgb|B|per_symbol|d3    val recall@b2 BTC 0.53 ETH 0.47 PR 0.418 rounds [147, 152] (123s)
xgb|B|per_symbol|d5    val recall@b2 BTC 0.70 ETH 0.53 PR 0.461 rounds [381, 349] (151s)


,feature_set,scope,max_depth,val_recall_btc,val_recall_eth,val_pr_auc,rounds,primary
config,,,,,,,,
xgb|A|pooled|d3,A,pooled,3,0.5,0.4333,0.3551,"[28, 28]",0.4667
xgb|A|pooled|d5,A,pooled,5,0.5667,0.5333,0.4043,"[15, 15]",0.55
xgb|A|per_symbol|d3,A,per_symbol,3,0.5333,0.4667,0.4223,"[439, 240]",0.5
xgb|A|per_symbol|d5,A,per_symbol,5,0.5333,0.4333,0.3941,"[19, 251]",0.4833
xgb|B|pooled|d3,B,pooled,3,0.7,0.6333,0.4989,"[698, 698]",0.6667
xgb|B|pooled|d5,B,pooled,5,0.7,0.6,0.5079,"[587, 587]",0.65
xgb|B|per_symbol|d3,B,per_symbol,3,0.5333,0.4667,0.4184,"[147, 152]",0.5
xgb|B|per_symbol|d5,B,per_symbol,5,0.7,0.5333,0.4612,"[381, 349]",0.6167


SELECTED (validation only): xgb|B|pooled|d3


## 6 · Final 3-seed ensemble, baselines, VALIDATION results (saved before the test split is unlocked)

In [7]:
fsel, scope_sel, depth_sel = SEL_CFG; FINAL = {s: [] for s in SYMS}; FINAL_LOG = []; TS['final_train_start'] = UTC()
for sd in SEEDS:
    if scope_sel == 'pooled':
        b, bi, info = train_booster(fsel, list(SYMS), depth_sel, sd, tag='final'); [FINAL[s].append((b, bi)) for s in SYMS]; FINAL_LOG.append({'scope': 'pooled', 'seed': sd, **info})
    else:
        for s in SYMS: b, bi, info = train_booster(fsel, [s], depth_sel, sd, tag='final|' + s); FINAL[s].append((b, bi)); FINAL_LOG.append({'scope': s, 'seed': sd, **info})
score_into('xgb_final', fsel, FINAL); TS['final_train_end'] = UTC(); print('final ensemble trained:', FINAL_LOG[0], '...')
# logistic-regression baseline on set A (same rows, class weight 5, standardised with TRAIN mean/std)
parts = [rows(s, 'train', 'A', None, 'logreg') for s in SYMS]; Xl = np.concatenate([p[0] for p in parts]); yl = np.concatenate([p[1] for p in parts]); mu, sdv = Xl.mean(axis=0), Xl.std(axis=0) + 1e-8
LR = LogisticRegression(C=1.0, max_iter=500, class_weight={0: 1.0, 1: 5.0}).fit((Xl - mu) / sdv, yl)
for sym in SYMS:
    for src, store in (('l1', SC1), ('l3', SC3)):
        s = LR.decision_function((np.nan_to_num(XF[(src, sym)][:, :NA]) - mu) / sdv).astype(np.float32); s[:FEAT_START] = np.nan; store[(sym, 'logreg_A')] = s
DECL = ['xgb_final', 'ae_recommended', 'logreg_A', 'rz']; register(['xgb_final', 'logreg_A'])
TS['val_eval_start'] = UTC(); RES_VAL = {f'{s}|{c}': l3_eval(s, 'val', c, ci=True) for s in SYMS for c in DECL}
dump_json({'protocol_sha256': PROTOCOL_HASH, 'selected': SELECTED, 'declared_set': DECL, 'thresholds': {f'{k[0]}|{k[1]}': v for k, v in THRB.items()}, 'val_results': RES_VAL, 'saved_utc': UTC()}, STAGE / 'reports' / 'validation_results.json'); TS['val_saved'] = UTC()
def comp_table(R, op):
    rows_ = {}
    for k, v in R.items():
        o = v['ops'][op]; rows_[k] = {'ROC': round(v['roc_auc'], 3), 'PR': round(v['pr_auc'], 3), 'chance': round(v['pr_auc_chance'], 3), 'recall': f"{o['events_detected']}/{v['n_events']} ({o['event_recall_ci95'][0]:.2f}-{o['event_recall_ci95'][1]:.2f})",
                      'delay': o['median_detection_delay_bars'], 'L1 ep/d': round(v['layer1'][op]['episodes_per_day'], 2), 'L3clean ep/d': round(o['clean_negative_episodes_per_day'], 2), 'fam': f"{o['families_covered']}/{o['families_total']}"}
    return pd.DataFrame(rows_).T
display(comp_table(RES_VAL, f'b{PB}'))

final ensemble trained: {'scope': 'pooled', 'seed': 11, 'n_train': 176580, 'n_train_pos': 4893, 'n_es': 35640, 'n_es_pos': 1736, 'best_iteration': 697, 'best_aucpr_val': 0.48025731371708447} ...


,ROC,PR,chance,recall,delay,L1 ep/d,L3clean ep/d,fam
BTCUSDT|xgb_final,0.839,0.468,0.05,22/30 (0.56-0.86),5.5,2,2.13,7/7
BTCUSDT|ae_recommended,0.717,0.181,0.05,16/30 (0.36-0.70),2,2,2.04,7/7
BTCUSDT|logreg_A,0.764,0.339,0.05,12/30 (0.25-0.58),9.5,2,1.96,5/7
BTCUSDT|rz,0.614,0.086,0.05,0/30 (0.00-0.11),None,0,0,0/7
ETHUSDT|xgb_final,0.854,0.536,0.047,19/30 (0.46-0.78),5,1.92,1.87,6/7
ETHUSDT|ae_recommended,0.727,0.156,0.047,11/30 (0.22-0.54),6,2,2.04,5/7
ETHUSDT|logreg_A,0.811,0.439,0.047,12/30 (0.25-0.58),8.5,2,2.12,6/7
ETHUSDT|rz,0.628,0.078,0.047,0/30 (0.00-0.11),None,0,0,0/7


## 7 · Leave-one-scenario-family-out: does the supervised model generalise to a mechanism it never saw?
Each fold retrains the selected configuration **without** every event containing the held-out family (composites included, early-stopping set included) and measures recall on that family's non-composite events from the train and validation splits. The test split stays locked.

In [8]:
base = EV3[EV3.composite == 0]; L2N = {}
for r in base.itertuples(): L2N.setdefault(r.fams[0], set()).add(r.scenario_family)
assert all(len(v) == 1 for v in L2N.values()) and len(L2N) == 6, L2N
L2N = {k: next(iter(v)) for k, v in L2N.items()}; EV3['fam_names'] = EV3.fams.apply(lambda l: sorted(L2N[x] for x in l)); print('family letters:', L2N)
def event_recall(sym, scorer, evs, op=f'b{PB}'):
    al = SC3[(sym, scorer)] > THRB[(sym, scorer)][int(op[1:])]; return int(sum(bool(al[r.start_idx:r.end_idx + 1].any()) for r in evs.itertuples())), len(evs)
LOFO, LOFO_LOG = {}, []; TS['lofo_start'] = UTC()
for fam in BASE_FAMS:
    excl = {}
    for sym in SYMS:
        m = np.zeros(N_BARS, bool)
        for r in EV3[(EV3.symbol == sym) & EV3.fam_names.apply(lambda l: fam in l)].itertuples(): m[r.start_idx:r.end_idx + 1] = True
        excl[sym] = m
    n0 = len(ROWLOG)
    if scope_sel == 'pooled': b, bi, info = train_booster(fsel, list(SYMS), depth_sel, 11, excl=excl, tag='lofo|' + fam); fold = {s: [(b, bi)] for s in SYMS}
    else:
        fold = {}
        for s in SYMS: b, bi, info = train_booster(fsel, [s], depth_sel, 11, excl=excl, tag='lofo|' + fam + '|' + s); fold[s] = [(b, bi)]
    nm = 'lofo_' + fam; score_into(nm, fsel, fold); register([nm]); fl = ROWLOG[n0:]
    for sym in SYMS:
        a, bnd = BOUNDS['train']; exp_pos = int(POS[sym][a:bnd].sum() - (POS[sym] & excl[sym])[a:bnd].sum()); got = sum(r['n_pos'] for r in fl if r['sym'] == sym and r['tag'].endswith('|fit'))
        LOFO_LOG.append({'family': fam, 'sym': sym, 'expected_train_pos_bars_without_family': exp_pos, 'train_pos_bars_in_fold_rows': int(got), 'excluded_pos_bars_total': int((POS[sym] & excl[sym]).sum())})
    tot = {'xgb_fold': [0, 0], 'ae': [0, 0], 'xgb_final_seen': [0, 0]}; per = {}
    for sym in SYMS:
        evs = EV3[(EV3.symbol == sym) & (EV3.composite == 0) & (EV3.scenario_family == fam) & EV3.split.isin(['train', 'val'])]
        for key, sc in (('xgb_fold', nm), ('ae', 'ae_recommended'), ('xgb_final_seen', 'xgb_final')):
            k, n = event_recall(sym, sc, evs); tot[key][0] += k; tot[key][1] += n; per[(sym, key)] = [k, n]
    LOFO[fam] = {'xgb_fold': tot['xgb_fold'], 'ae': tot['ae'], 'xgb_final_trained_on_family': tot['xgb_final_seen'], 'per_symbol': {f'{s}|{k}': v for (s, k), v in per.items()}}
    LOFO[fam]['thr_b%d' % PB] = {s: THRB[(s, nm)][PB] for s in SYMS}
    for sym in SYMS: del SC1[(sym, nm)], SC3[(sym, nm)], THRB[(sym, nm)]
    print(f"held out {fam:28s} xgb {tot['xgb_fold'][0]}/{tot['xgb_fold'][1]} | AE {tot['ae'][0]}/{tot['ae'][1]} | (final model, family seen in training: {tot['xgb_final_seen'][0]}/{tot['xgb_final_seen'][1]})")
LOFO_MEAN = {k: float(np.mean([LOFO[f][k][0] / max(LOFO[f][k][1], 1) for f in BASE_FAMS])) for k in ('xgb_fold', 'ae', 'xgb_final_trained_on_family')}
LOFO_STRENGTH = 'strong' if LOFO_MEAN['xgb_fold'] >= 0.6 else ('moderate' if LOFO_MEAN['xgb_fold'] >= 0.3 else 'weak'); print('macro mean recall:', {k: round(v, 3) for k, v in LOFO_MEAN.items()}, '->', LOFO_STRENGTH); TS['lofo_done'] = UTC()

family letters: {'D': 'trade_flow_imbalance', 'A': 'pump_and_dump_like', 'C': 'abnormal_volume_burst', 'B': 'fake_breakout_like', 'E': 'wash_like_aggregate_proxy', 'F': 'liquidity_stress_proxy'}
held out pump_and_dump_like           xgb 20/34 | AE 10/34 | (final model, family seen in training: 28/34)
held out fake_breakout_like           xgb 18/34 | AE 9/34 | (final model, family seen in training: 26/34)
held out abnormal_volume_burst        xgb 5/34 | AE 6/34 | (final model, family seen in training: 12/34)
held out trade_flow_imbalance         xgb 7/32 | AE 2/32 | (final model, family seen in training: 17/32)
held out wash_like_aggregate_proxy    xgb 2/32 | AE 11/32 | (final model, family seen in training: 24/32)
held out liquidity_stress_proxy       xgb 27/32 | AE 16/32 | (final model, family seen in training: 30/32)
macro mean recall: {'xgb_fold': 0.398, 'ae': 0.274, 'xgb_final_trained_on_family': 0.693} -> moderate


## 8 · Exact TreeSHAP (validation bars): which feature groups drive the score, and for which scenario

In [9]:
GROUPS = FEATURES + ['cross', 'ae']
def group_of(n): return n.split('__')[0] if n.split('__')[0] in FEATURES else ('cross' if n.startswith('x__') else 'ae')
GIDX = {g: [i for i, n in enumerate(NAMES[fsel]) if group_of(n) == g] for g in GROUPS}; rng = np.random.default_rng(0); SH_ROWS, SH_FAM, SH_PM = [], [], []
a, b = BOUNDS['val']
for sym in SYMS:
    idx = np.arange(N_BARS); pos_i = np.flatnonzero((idx >= a) & (idx < b) & POS[sym] & (idx >= FEAT_START)); cl_i = np.flatnonzero((idx >= a) & (idx < b) & CLEAN[sym]); cl_i = rng.choice(cl_i, min(3000, len(cl_i)), replace=False); ii = np.concatenate([pos_i, cl_i])
    X = XF[('l3', sym)][ii][:, :NCOL[fsel]]; dm = xgb.DMatrix(X, feature_names=NAMES[fsel]); C = np.mean([bst.predict(dm, pred_contribs=True, iteration_range=(0, bi + 1)) for bst, bi in FINAL[sym]], axis=0)
    mg = np.mean([bst.predict(dm, output_margin=True, iteration_range=(0, bi + 1)) for bst, bi in FINAL[sym]], axis=0); SH_PM.append(float(np.abs(C.sum(axis=1) - mg).max()))
    G = np.column_stack([C[:, GIDX[g]].sum(axis=1) for g in GROUPS]); lab = LB3[sym]
    for j, i in enumerate(ii): SH_ROWS.append((sym, i, bool(POS[sym][i]), G[j], C[j, :-1]))
    for fam in BASE_FAMS:
        m = lab['label_' + fam].to_numpy()[ii] == 1
        if m.any(): SH_FAM.append((fam, G[m]))
SHAP_MAXERR = max(SH_PM)
Gpos = np.array([r[3] for r in SH_ROWS if r[2]]); Gneg = np.array([r[3] for r in SH_ROWS if not r[2]]); Fpos = np.array([r[4] for r in SH_ROWS if r[2]])
SHAP_REPORT = {'additivity_max_abs_error': SHAP_MAXERR, 'n_pos_bars': int(len(Gpos)), 'n_clean_bars': int(len(Gneg)), 'group_mean_abs_shap_positive': dict(zip(GROUPS, np.abs(Gpos).mean(axis=0).tolist())), 'group_mean_abs_shap_clean': dict(zip(GROUPS, np.abs(Gneg).mean(axis=0).tolist())),
               'top_features_positive': sorted(zip(NAMES[fsel], np.abs(Fpos).mean(axis=0).tolist()), key=lambda x: -x[1])[:12],
               'by_family_group_mean_abs': {f: dict(zip(GROUPS, np.abs(np.concatenate([g for ff, g in SH_FAM if ff == f])).mean(axis=0).tolist())) for f in BASE_FAMS if any(ff == f for ff, _ in SH_FAM)}}
display(pd.DataFrame(SHAP_REPORT['by_family_group_mean_abs']).T.round(3)); print('top features:', [(n, round(v, 3)) for n, v in SHAP_REPORT['top_features_positive'][:8]]); print('TreeSHAP additivity max error:', SHAP_MAXERR); TS['shap_done'] = UTC()

,log_returns,trade_flow_imbalance,hl_spread_ratio,rush_order_density,volume_acceleration,large_trade_count,cross,ae
pump_and_dump_like,0.802,0.839,2.346,0.402,0.134,0.409,0.156,1.106
fake_breakout_like,0.902,0.469,2.448,0.415,0.141,0.342,0.175,1.122
abnormal_volume_burst,0.732,0.708,1.407,0.378,0.225,0.426,0.176,0.997
trade_flow_imbalance,0.638,1.075,1.717,0.451,0.16,0.296,0.177,0.799
wash_like_aggregate_proxy,0.639,0.576,0.782,0.854,0.124,0.215,0.094,0.875
liquidity_stress_proxy,0.605,1.437,2.334,0.462,0.118,0.244,0.121,1.217


top features: [('hl_spread_ratio__mean15', 1.108), ('trade_flow_imbalance__amax15', 0.554), ('log_returns__std15', 0.548), ('ae__e_large_trade_count', 0.441), ('ae__fmax', 0.346), ('ae__fmean', 0.33), ('hl_spread_ratio__mean3', 0.322), ('rush_order_density__mean15', 0.32)]
TreeSHAP additivity max error: 4.291534423828125e-06


## 9 · UNLOCK THE TEST SPLIT (once, declared set only)

In [10]:
assert (STAGE / 'reports' / 'validation_results.json').is_file() and not TEST_STATE['unlocked']
TEST_STATE['unlocked'] = True; TEST_STATE['unlock_utc'] = UTC(); TS['test_unlocked'] = TEST_STATE['unlock_utc']
RES_TEST = {f'{s}|{c}': l3_eval(s, 'test', c, ci=True) for s in SYMS for c in DECL}; TEST_STATE['unlocked'] = False; TS['test_evaluated'] = UTC()
print('TEST evaluated once for the declared set.'); display(comp_table(RES_TEST, f'b{PB}'))
DRIFT = {k: {'val_ep_day': RES_VAL[k]['layer1'][f'b{PB}']['episodes_per_day'], 'test_ep_day': RES_TEST[k]['layer1'][f'b{PB}']['episodes_per_day'], 'DRIFT_FLAG': bool(RES_TEST[k]['layer1'][f'b{PB}']['episodes_per_day'] > 3 * PB)} for k in RES_TEST}

TEST evaluated once for the declared set.


,ROC,PR,chance,recall,delay,L1 ep/d,L3clean ep/d,fam
BTCUSDT|xgb_final,0.869,0.609,0.088,26/36 (0.56-0.84),11,0.5,0.58,7/7
BTCUSDT|ae_recommended,0.732,0.272,0.088,18/36 (0.34-0.66),10.5,1.64,1.32,7/7
BTCUSDT|logreg_A,0.798,0.455,0.088,15/36 (0.27-0.58),17,0.43,0.41,7/7
BTCUSDT|rz,0.629,0.139,0.088,0/36 (0.00-0.10),None,0,0,0/7
ETHUSDT|xgb_final,0.813,0.506,0.066,24/36 (0.50-0.80),17.5,1.29,1.21,7/7
ETHUSDT|ae_recommended,0.703,0.183,0.066,10/36 (0.16-0.44),20.5,0.79,0.81,6/7
ETHUSDT|logreg_A,0.748,0.392,0.066,18/36 (0.34-0.66),15,0.5,0.57,5/7
ETHUSDT|rz,0.616,0.101,0.066,0/36 (0.00-0.10),None,0,0,0/7


## 10 · Persist models, scores, reports and manifest (staging only)

In [11]:
MODELDIR = {}
for sym in SYMS:
    for sd, (bst, bi) in zip(SEEDS, FINAL[sym]):
        key = 'pooled' if scope_sel == 'pooled' else SYMS[sym]
        if scope_sel == 'pooled' and sym != 'BTCUSDT': continue
        p = STAGE / 'models' / key; p.mkdir(parents=True, exist_ok=True); bst.save_model(str(p / f'seed{sd}.json')); MODELDIR.setdefault(key, {})[sd] = bi + 1
for sym in SYMS:
    dump_json({'features': FEATURES, 'log1p_columns': ['rush_order_density', 'large_trade_count'], 'median': PREP[sym]['median'], 'scale': PREP[sym]['scale'], 'clip': [-20, 20], 'fit_rows': PREP[sym]['fit_rows'], 'source': 'Layer 1 TRAIN only (identical to the Autoencoder)'}, STAGE / 'preprocessing' / f'{SYMS[sym]}_preprocessing.json')
    for src, store in (('l1', SC1), ('l3', SC3)):
        pd.DataFrame({'datetime': DT[sym], 'score': store[(sym, 'xgb_final')].astype(np.float32), 'prob_uncalibrated': (1 / (1 + np.exp(-store[(sym, 'xgb_final')].astype(np.float64)))).astype(np.float32)}).to_parquet(STAGE / 'scores' / f'{SYMS[sym]}_xgb_final_scores_{src}.parquet', index=False)
dump_json({'selected_config': SELECTED, 'feature_set': fsel, 'feature_names': NAMES[fsel], 'scope': scope_sel, 'max_depth': depth_sel, 'params': PARAMS, 'rounds_used': MODELDIR, 'seeds': SEEDS, 'score': 'mean raw margin over seeds (log-odds)', 'ae_dependency': (AE_SCORER + ' from models_autoencoder_v2 (frozen)') if fsel == 'B' else None,
           'thresholds_episodes_per_day_budget': {SYMS[s]: {str(b_): t for b_, t in THRB[(s, 'xgb_final')].items()} for s in SYMS}, 'threshold_source': 'genuine Layer 1 VALIDATION'}, STAGE / 'models' / 'selected_config.json')
dump_json({f'{k[0]}|{k[1]}': {str(b_): t for b_, t in v.items()} for k, v in THRB.items()}, STAGE / 'models' / 'thresholds_by_budget.json')
dump_json({'selection_table': SELTAB.assign(simp=SELTAB.simp.apply(list)).to_dict(orient='records'), 'selected': SELECTED, 'rule': PROTOCOL['selection_rule']}, STAGE / 'reports' / 'selection_table.json')
dump_json({'lofo': LOFO, 'lofo_macro_mean_recall': LOFO_MEAN, 'lofo_strength': LOFO_STRENGTH, 'lofo_row_log': LOFO_LOG}, STAGE / 'reports' / 'lofo_report.json'); dump_json(SHAP_REPORT, STAGE / 'reports' / 'shap_report.json')
dump_json({'selection_training': TRAIN_LOG, 'final_training': FINAL_LOG, 'row_log': ROWLOG, 'timestamps': TS, 'xgboost': xgb.__version__, 'params': PARAMS}, STAGE / 'reports' / 'training_report.json')
dump_json({'protocol_sha256': PROTOCOL_HASH, 'validation': RES_VAL, 'test': RES_TEST, 'layer1_drift': DRIFT, 'test_unlock_utc': TEST_STATE['unlock_utc'], 'test_evaluation_calls': TEST_STATE['calls']}, STAGE / 'reports' / 'evaluation_report.json')
LIM = ['Layer 3 is synthetic: labels are injection mechanisms in aggregate data. A high score means "resembles an injected mechanism", not "manipulation happened". Spoofing, layering and liquidity withdrawal are not represented.',
       'The supervised model can learn generator-specific signatures. The leave-one-family-out table and the OOD events are the only evidence about unseen mechanisms; both are modest in size and come from the same generator.',
       'Selection among 8 configurations used 30 validation events per symbol and the same split provided early stopping, so validation numbers are slightly optimistic. Test was unlocked once after validation was saved.',
       'Set-B features depend on the Autoencoder (trained on Layer 1 TRAIN, in-sample on train bars): the AE error on training bars is lower than out-of-sample. This covariate shift is why set A and set B were both candidates.',
       'Thresholds come from ~13 genuine validation days (13-52 episodes). Layer 1 test drift can change test alarm rates (DRIFT flag in the report).',
       'Scores are uncalibrated log-odds; probabilities in the parquet are not real probabilities. Conformal calibration is a later stage. Layer 1 raw-tick reconciliation of two features is still pending.',
       'Negatives are clean bars only; bars within 15 bars after an injection are excluded from both training and evaluation, so post-event behaviour is not assessed.']
(STAGE / 'reports' / 'limitations.md').write_text('# Limitations\n\n' + '\n'.join('- ' + l for l in LIM) + '\n', encoding='utf-8')
(STAGE / 'README.md').write_text('\n'.join(['# models_xgboost_v1', '', 'Supervised XGBoost pattern classifier for Component 03. protocol/xgboost_protocol.json was frozen (sha256 ' + PROTOCOL_HASH + ') before any feature or model existed.', '',
    '- models/selected_config.json: configuration, feature names, rounds, thresholds per alarm-episode budget (1/2/4 per day)', '- models/{pooled|btc|eth}/seed*.json: XGBoost boosters (score = mean raw margin over seeds)', '- scores/*_xgb_final_scores_{l1,l3}.parquet', '- reports/: selection_table, validation_results, evaluation_report (test, exploratory comparators), lofo_report, shap_report, training_report, limitations.md', '', 'Read reports/limitations.md before quoting any number.']), encoding='utf-8')
def build_manifest():
    return {str(p.relative_to(STAGE)): {'size': p.stat().st_size, 'sha256': sha256_file(p)} for p in sorted(STAGE.rglob('*')) if p.is_file() and p.name not in ('manifest.json', 'validation_report.json')}
MANIFEST = {'artifact': 'models_xgboost_v1', 'protocol_sha256': PROTOCOL_HASH, 'excluded': ['manifest.json', 'validation_report.json'], 'files': build_manifest()}; dump_json(MANIFEST, STAGE / 'manifest.json'); print('manifest entries:', len(MANIFEST['files']))

manifest entries: 20


## 11 · Validation of the NEW artifacts only, publish, and SHARE BLOCK

In [12]:
RES = []
def T(n, name, ok, detail=''):
    RES.append({'test': n, 'name': name, 'pass': bool(ok), 'detail': str(detail)[:300]}); print(('PASS ' if ok else 'FAIL ') + f'T{n:02d} {name}' + ('' if ok else f'  <-- {str(detail)[:200]}'))
fname = lambda sc: sc.replace('|', '_')
req = ['protocol/xgboost_protocol.json', 'models/selected_config.json', 'models/thresholds_by_budget.json', 'reports/selection_table.json', 'reports/validation_results.json', 'reports/evaluation_report.json', 'reports/lofo_report.json', 'reports/shap_report.json', 'reports/training_report.json', 'reports/limitations.md', 'README.md', 'manifest.json'] + \
      [f'preprocessing/{SYMS[s]}_preprocessing.json' for s in SYMS] + [f'scores/{SYMS[s]}_xgb_final_scores_{src}.parquet' for s in SYMS for src in ('l1', 'l3')] + [f'models/{k}/seed{sd}.json' for k in MODELDIR for sd in SEEDS]
T(1, 'all required files exist', all((STAGE / r).is_file() for r in req), [r for r in req if not (STAGE / r).is_file()][:5])
T(2, 'frozen sources unchanged (Layers 1, 2, 3 v1/v2, Autoencoder v1/v2)', {n: pin_dir(BASE / n) for n in PIN} == PIN)
pj = json.loads((STAGE / 'protocol' / 'xgboost_protocol.json').read_text()); body = {k: v for k, v in pj.items() if k != 'integrity'}
order = ['protocol_frozen', 'data_loaded', 'features_built', 'selection_start', 'selection_done', 'final_train_end', 'val_saved', 'lofo_done', 'shap_done', 'test_unlocked', 'test_evaluated']
T(3, 'protocol intact (hash matches) and frozen BEFORE any feature/model; selection and LOFO only on train/validation; test unlocked last', hashlib.sha256(json.dumps(body, sort_keys=True, separators=(',', ':'), ensure_ascii=False).encode()).hexdigest() == PROTOCOL_HASH == pj['integrity']['sha256_of_canonical_json_excluding_this_field'] and all(TS[a_] <= TS[b_] for a_, b_ in zip(order, order[1:])), {k: TS[k] for k in order})
ok4 = True
for sym in SYMS:
    X = np.asarray(F1[sym], float).copy(); X[:, 3] = np.log1p(np.maximum(X[:, 3], 0)); X[:, 5] = np.log1p(np.maximum(X[:, 5], 0)); tr = X[15:90720]; med = np.median(tr, axis=0); iq = np.subtract(*np.percentile(tr, [75, 25], axis=0)) / 1.349; sc_ = np.maximum(np.where(iq > 1e-12, iq, tr.std(axis=0)), 1e-8)
    pp = json.loads((STAGE / 'preprocessing' / f'{SYMS[sym]}_preprocessing.json').read_text()); ok4 &= bool(np.allclose(pp['median'], med, rtol=1e-9) and np.allclose(pp['scale'], sc_, rtol=1e-9) and pp['fit_rows'] == [15, 90720])
T(4, 'preprocessing fitted on Layer 1 TRAIN bars only (independent numpy recompute; identical to the Autoencoder files)', ok4)
ok5 = True
for sym, tag in SYMS.items():
    ref = pd.read_parquet(AE2 / 'scores' / f'{tag}_{AE_SCORER.replace("|", "_")}_scores_l3.parquet')['score'].to_numpy(); new = SC3[(sym, 'ae_recommended')]; m = np.isfinite(ref) & np.isfinite(new); ok5 &= bool(m.sum() > 100000 and np.allclose(new[m], ref[m], rtol=2e-3, atol=1e-5))
T(5, 'recomputed Autoencoder comparator scores reproduce the saved Autoencoder v2 Layer 3 scores', ok5)
ok6 = True
for sym in SYMS:
    Zs = Z1[sym][:6000].copy(); Zq = Zs.copy(); Zq[3000:] += 5.0; fa, _ = build_feats(Zs); fb, _ = build_feats(Zq); ok6 &= bool(np.allclose(fa[:3000], fb[:3000], equal_nan=True, atol=1e-6) and not np.allclose(fa[3000:], fb[3000:], equal_nan=True))
    aa, ab = ae_cols(sym, Zs), ae_cols(sym, Zq); ok6 &= bool(np.allclose(aa[:3000], ab[:3000], equal_nan=True, atol=1e-6) and not np.allclose(aa[3000:], ab[3000:], equal_nan=True))
T(6, 'features and Autoencoder columns are causal (changing bars after t leaves values at <= t unchanged)', ok6)
ok7 = True
for sym in SYMS:
    m = CLEAN[sym]; ok7 &= bool(np.allclose(XF[('l1', sym)][m], XF[('l3', sym)][m], rtol=1e-4, atol=1e-4, equal_nan=True))
T(7, 'features of clean bars are identical in genuine Layer 1 and Layer 3 (no injection leaks into negatives)', ok7)
ok8 = True
for r in ROWLOG:
    if '|fit' in r['tag'] or r['tag'].startswith('logreg'): ok8 &= bool(r['min_idx'] >= FEAT_START and r['max_idx'] < BOUNDS['train'][1])
    elif '|es' in r['tag']: ok8 &= bool(r['min_idx'] >= BOUNDS['val'][0] and r['max_idx'] < BOUNDS['val'][1])
    else: ok8 = False
T(8, 'every fitting row is inside Layer 3 TRAIN; every early-stopping row inside VALIDATION; no test row used for fitting', ok8 and len(ROWLOG) > 20)
ok9 = True; rr = np.random.default_rng(5)
for sym in SYMS:
    nb = LB3[sym].normal_background.to_numpy()
    for t in rr.integers(0, N_BARS, 3000): ok9 &= bool(CLEAN[sym][t] == (t >= FEAT_START and all(nb[max(t - 15, 0):t + 1] == 1)))
T(9, 'clean-bar mask independently recomputed (plain loop on 6000 random bars) matches', ok9)
ok10 = True; days_ = (BOUNDS['val'][1] - BOUNDS['val'][0]) / 1440; thj = json.loads((STAGE / 'models' / 'thresholds_by_budget.json').read_text())
for (sym, sc), tb in THRB.items():
    v = SC1[(sym, sc)][BOUNDS['val'][0]:BOUNDS['val'][1]]
    for b_, t in tb.items():
        idx = np.flatnonzero(v > t); n_ep = 0 if len(idx) == 0 else 1 + sum(1 for i in range(1, len(idx)) if idx[i] - idx[i - 1] > 5); ok10 &= bool(n_ep / days_ <= b_ + 1e-9 and np.isclose(thj[f'{sym}|{sc}'][str(b_)], t))
        low = np.unique(np.quantile(v[np.isfinite(v)], PGRID)); low = low[low < t]; ok10 &= bool(all(episodes(v > x) / days_ > b_ for x in low))
T(10, 'every budget threshold recomputed from genuine Layer 1 VALIDATION: episodes/day <= budget and minimal', ok10)
tab = pd.DataFrame(json.loads((STAGE / 'reports' / 'selection_table.json').read_text())['selection_table']); tab['simp'] = tab.simp.apply(tuple)
T(11, 'configuration selection recomputed from the saved validation table alone', select_rule(tab) == SELECTED == json.loads((STAGE / 'models' / 'selected_config.json').read_text())['selected_config'] and len(tab) == 8)
T(12, 'test split evaluated in exactly one unlocked pass, declared set only, after validation results were saved', TEST_STATE['calls'] == len(SYMS) * len(DECL) and set(RES_TEST) == {f'{s}|{c}' for s in SYMS for c in DECL} and TS['val_saved'] < TS['test_unlocked'] < TS['test_evaluated'])
ok13 = True
for sym in SYMS:
    sdf = pd.read_parquet(STAGE / 'scores' / f'{SYMS[sym]}_xgb_final_scores_l3.parquet')['score'].to_numpy()
    for sp, R in (('val', RES_VAL), ('test', RES_TEST)):
        for op, o in R[f'{sym}|xgb_final']['ops'].items():
            evs = EV3[(EV3.symbol == sym) & (EV3.split == sp)]; det = 0
            for r in evs.itertuples():
                hit = False
                for i in range(int(r.start_idx), int(r.end_idx) + 1):
                    if sdf[i] > o['threshold']: hit = True; break
                det += hit
            ok13 &= bool(det == o['events_detected'])
T(13, 'event recall of the final model independently recomputed (plain loop on stored scores), validation and test, every budget', ok13)
b2_, bi2_, _ = train_booster(fsel, list(SYMS) if scope_sel == 'pooled' else ['BTCUSDT'], depth_sel, SEEDS[0], tag='repro'); sym0 = 'BTCUSDT'; s_new = margin([(b2_, bi2_)], fsel, XF[('l3', sym0)]); s_old = margin([FINAL[sym0][0]], fsel, XF[('l3', sym0)]); vv = slice(BOUNDS['val'][0], BOUNDS['val'][1])
sp_ = stats.spearmanr(s_new[vv], s_old[vv])[0]; T(14, 'retraining the first final seed reproduces its scores (Spearman >= 0.999, same number of rounds)', sp_ >= 0.999 and bi2_ == FINAL[sym0][0][1], f'spearman {sp_:.6f}, rounds {bi2_}/{FINAL[sym0][0][1]}')
T(15, 'TreeSHAP is additive: sum of contributions equals the model margin (max abs error < 1e-3)', SHAP_MAXERR < 1e-3, SHAP_MAXERR)
T(16, 'leave-one-family-out folds trained without the held-out family (fold positive-bar counts equal the full count minus the family\'s bars)', all(r['train_pos_bars_in_fold_rows'] == r['expected_train_pos_bars_without_family'] and r['excluded_pos_bars_total'] > 0 for r in LOFO_LOG) and len(LOFO) == 6, [r for r in LOFO_LOG if r['train_pos_bars_in_fold_rows'] != r['expected_train_pos_bars_without_family']][:2])
T(17, 'all stored scores finite after the warm-up', all(np.isfinite(pd.read_parquet(STAGE / 'scores' / f'{SYMS[s]}_xgb_final_scores_{src}.parquet').score.to_numpy()[100:]).all() for s in SYMS for src in ('l1', 'l3')))
man = json.loads((STAGE / 'manifest.json').read_text())['files']; actual = {str(p.relative_to(STAGE)) for p in STAGE.rglob('*') if p.is_file() and p.name not in ('manifest.json', 'validation_report.json')}
T(18, 'manifest lists every file with correct sha256/size', set(man) == actual and all(sha256_file(STAGE / k) == v['sha256'] for k, v in man.items()), sorted(actual ^ set(man))[:5])
for p in STAGE.rglob('*.json'): json.loads(p.read_text())
for p in STAGE.rglob('*.parquet'): pq.read_metadata(p)
T(19, 'all JSON/parquet outputs readable', True)
ALL_PASS = all(r['pass'] for r in RES)
rc = lambda R, sc, op=f'b{PB}': float(np.mean([R[f'{s}|{sc}']['ops'][op]['event_recall'] for s in SYMS]))
D_VAL, D_TEST = rc(RES_VAL, 'xgb_final') - rc(RES_VAL, 'ae_recommended'), rc(RES_TEST, 'xgb_final') - rc(RES_TEST, 'ae_recommended'); DRIFTED = any(v['DRIFT_FLAG'] for k, v in DRIFT.items() if k.endswith('xgb_final'))
if not ALL_PASS: VERDICT = 'BLOCKED'
elif D_VAL >= 0.05 and D_TEST >= 0.05: VERDICT = 'XGBOOST OUTPERFORMS the Autoencoder at matched burden on validation and test (synthetic world)'
elif abs(D_VAL) < 0.05 and abs(D_TEST) < 0.05: VERDICT = 'XGBOOST COMPARABLE to the Autoencoder at matched burden (no significant difference)'
elif D_VAL <= -0.05 or D_TEST <= -0.05: VERDICT = 'XGBOOST DOES NOT OUTPERFORM the Autoencoder at matched burden'
else: VERDICT = 'MIXED: XGBoost advantage not consistent across validation and test'
VERDICT += f'; generalisation to unseen scenario families: {LOFO_STRENGTH}' + ('; Layer 1 TEST BURDEN DRIFT FLAGGED' if DRIFTED else '')
print(f"\n{sum(r['pass'] for r in RES)}/{len(RES)} tests passed | {VERDICT}")
dump_json({'tests': RES, 'verdict': VERDICT, 'protocol_sha256': PROTOCOL_HASH, 'run_started_utc': RUN_STARTED}, STAGE / 'validation' / 'validation_report.json')
assert {n: pin_dir(BASE / n) for n in PIN} == PIN, 'frozen inputs changed during the run'
if ALL_PASS: os.rename(STAGE, OUT_FINAL); where = OUT_FINAL; print('PUBLISHED ->', where)
else: where = STAGE; print('NOT PUBLISHED; staging kept:', where)

def tab_rows(R, op):
    out = {}
    for sym in SYMS:
        for sc in DECL:
            v = R[f'{sym}|{sc}']; o = v['ops'][op]; out[f'{sym[:3]}|{sc}'] = {'ROC': round(v['roc_auc'], 3), 'PR': round(v['pr_auc'], 3), 'chance': round(v['pr_auc_chance'], 3), 'recall': f"{o['events_detected']}/{v['n_events']} ({o['event_recall_ci95'][0]:.2f}-{o['event_recall_ci95'][1]:.2f})", 'delay': o['median_detection_delay_bars'], 'L1 ep/d': round(v['layer1'][op]['episodes_per_day'], 2), 'L3clean ep/d': round(o['clean_negative_episodes_per_day'], 2), 'fam': f"{o['families_covered']}/{o['families_total']}"}
    return pd.DataFrame(out).T
def fam_tab(R, op): return pd.DataFrame({f'{sym[:3]}|{sc}': {f: f'{a_}/{n_}' for f, (a_, n_) in R[f'{sym}|{sc}']['ops'][op]['by_family'].items()} for sym in SYMS for sc in ('xgb_final', 'ae_recommended')}).to_string()
print('\n==================== XGBOOST — FINAL SHARE BLOCK ====================')
print('1. Output:', where, '| protocol sha256:', PROTOCOL_HASH[:16], '(frozen before any feature/model) | device: CPU xgboost', xgb.__version__)
print('2. Pipeline tests:', f"{sum(r['pass'] for r in RES)}/{len(RES)}", '| positives/clean bars in TRAIN:', {s: (int(POS[s][:90720].sum()), int(CLEAN[s][:90720].sum())) for s in SYMS})
print('3. Selection on Layer 3 VALIDATION (8 configs, recall @ budget', PB, 'episodes/day):'); print(SELTAB.drop(columns=['simp', 'feature_set', 'scope', 'max_depth']).set_index('config').round(3).to_string()); print('   SELECTED:', SELECTED, '| final rounds:', MODELDIR)
print('4. VALIDATION @ matched budget', PB, 'episodes/day (Layer 3 events, 95% CI):'); print(tab_rows(RES_VAL, f'b{PB}').to_string())
print('5. TEST @ matched budget', PB, '(unlocked once; XGBoost never used it before):'); print(tab_rows(RES_TEST, f'b{PB}').to_string())
print('6. Recall at other budgets (mean of BTC/ETH): ', {sc: {f'b{b_}': (round(rc(RES_VAL, sc, f'b{b_}'), 3), round(rc(RES_TEST, sc, f'b{b_}'), 3)) for b_ in BUDGETS} for sc in DECL}, '(val, test)')
print('7. Event recall by scenario @ budget', PB, '(VALIDATION):'); print(fam_tab(RES_VAL, f'b{PB}')); print('   (TEST):'); print(fam_tab(RES_TEST, f'b{PB}'))
print('8. Genuine Layer 1 alarm episodes/day (val | test) and drift flag:', {k: (round(v['val_ep_day'], 2), round(v['test_ep_day'], 2), v['DRIFT_FLAG']) for k, v in DRIFT.items()})
print('9. OOD events @ budget', PB, '(TEST, xgb vs AE):', {f'{s[:3]}|{c}': RES_TEST[f'{s}|{c}']['ops'][f'b{PB}']['by_ood'] for s in SYMS for c in ('xgb_final', 'ae_recommended')}); print('   by severity (TEST, xgb):', {s[:3]: RES_TEST[f'{s}|xgb_final']['ops'][f'b{PB}']['by_severity'] for s in SYMS})
print('10. Leave-one-family-out (unseen mechanism; train+val events):'); print(pd.DataFrame({f: {'xgb (family unseen)': '%d/%d' % tuple(v['xgb_fold']), 'AE': '%d/%d' % tuple(v['ae']), 'xgb (family seen)': '%d/%d' % tuple(v['xgb_final_trained_on_family'])} for f, v in LOFO.items()}).T.to_string()); print('    macro mean recall:', {k: round(v, 3) for k, v in LOFO_MEAN.items()}, '->', LOFO_STRENGTH)
print('11. SHAP (validation positives) mean |SHAP| by feature group:', {k: round(v, 3) for k, v in SHAP_REPORT['group_mean_abs_shap_positive'].items()}); print('    clean bars:', {k: round(v, 3) for k, v in SHAP_REPORT['group_mean_abs_shap_clean'].items()}); print('    top features:', [(n, round(v, 3)) for n, v in SHAP_REPORT['top_features_positive'][:6]])
print(pd.DataFrame(SHAP_REPORT['by_family_group_mean_abs']).T.round(3).to_string())
print('12. Limitations: synthetic labels (mechanisms, not proven manipulation); possible generator-specific learning; selection+early stopping share the validation split; AE columns in-sample on train bars; thresholds from ~13 genuine days; scores uncalibrated; no spoofing/layering.')
print('13. FINAL VERDICT:', VERDICT)
print('=====================================================================')
if not ALL_PASS: raise AssertionError('validation failed; nothing published')

PASS T01 all required files exist
PASS T02 frozen sources unchanged (Layers 1, 2, 3 v1/v2, Autoencoder v1/v2)
PASS T03 protocol intact (hash matches) and frozen BEFORE any feature/model; selection and LOFO only on train/validation; test unlocked last
PASS T04 preprocessing fitted on Layer 1 TRAIN bars only (independent numpy recompute; identical to the Autoencoder files)
PASS T05 recomputed Autoencoder comparator scores reproduce the saved Autoencoder v2 Layer 3 scores
PASS T06 features and Autoencoder columns are causal (changing bars after t leaves values at <= t unchanged)
PASS T07 features of clean bars are identical in genuine Layer 1 and Layer 3 (no injection leaks into negatives)
PASS T08 every fitting row is inside Layer 3 TRAIN; every early-stopping row inside VALIDATION; no test row used for fitting
PASS T09 clean-bar mask independently recomputed (plain loop on 6000 random bars) matches
PASS T10 every budget threshold recomputed from genuine Layer 1 VALIDATION: episodes/day 